# Lab 06 External V2 — 07 Final Validation

**Dataset:** Synthea Healthcare  
**Architecture:** External Delta tables  
**Compute:** Databricks Serverless compatible

## Purpose

Validate all recurring Gold external tables end to end: existence, locations,
dimension keys, fact grains, foreign keys, aggregate grains, reconciliation,
and basic business sanity checks.

> Serverless compatibility rule: this notebook does **not** call
> `REFRESH TABLE`, `CACHE TABLE`, `UNCACHE TABLE`, or Spark cache-refresh APIs.


## 1. Runtime context

In [ ]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "REPLACE_WITH_EXTERNAL_GOLD_ROOT",
    "05 External Gold root",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

if not external_gold_root or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT":
    raise ValueError(
        "external_gold_root must be passed by lab06_00_dev_runner "
        "or entered manually."
    )

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError("external_gold_root must be an abfss:// path.")

source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
source_csv_path = f"{source_volume_path}/source/csv"
reference_path = f"{source_volume_path}/reference"
target_schema_fqn = f"{catalog}.{target_schema}"

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Run validation     : {run_validation}")

## 2. Helpers and expected external objects

In [ ]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.external_tables import (
    normalize_location,
    overwrite_external_delta,
    registered_table_location,
    register_external_delta_table,
    validate_registered_location,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}")

print("Serverless-safe external-table helpers loaded.")

DIMENSIONS = {
    "dim_date": "date_key",
    "dim_patient": "patient_key",
    "dim_provider": "provider_key",
    "dim_organization": "organization_key",
    "dim_payer": "payer_key",
    "dim_condition": "condition_key",
}

FACTS = {
    "fact_encounters": "encounter_key",
    "fact_conditions": "condition_event_key",
}

AGGREGATES = {
    "agg_daily_encounters": ["date_key"],
    "agg_organization_performance": ["organization_key"],
    "agg_payer_performance": ["payer_key"],
    "agg_condition_summary": ["condition_key"],
}

ALL_OBJECTS = list(DIMENSIONS) + list(FACTS) + list(AGGREGATES)

object_rows = []
missing_tables = []
location_failures = []

for object_name in ALL_OBJECTS:
    table_name = f"{target_schema_fqn}.{object_name}"
    expected_location = f"{external_gold_root}/{object_name}"
    exists = spark.catalog.tableExists(table_name)

    actual_location = (
        registered_table_location(spark, table_name)
        if exists else None
    )

    location_ok = (
        exists
        and actual_location is not None
        and normalize_location(actual_location)
        == normalize_location(expected_location)
    )

    if not exists:
        missing_tables.append(table_name)
    if exists and not location_ok:
        location_failures.append(object_name)

    object_rows.append(
        (
            object_name,
            table_name,
            actual_location,
            "PASS" if exists and location_ok else "FAIL",
        )
    )

display(spark.createDataFrame(
    object_rows,
    ["object_name","table_name","actual_location","status"],
))

if missing_tables:
    raise RuntimeError("Missing required tables: " + ", ".join(missing_tables))

## 3. Dimension key validation

In [ ]:
dimension_failures = []
dimension_rows = []

for object_name, key_col in DIMENSIONS.items():
    table_name = f"{target_schema_fqn}.{object_name}"
    df = spark.table(table_name)
    p = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(key_col).alias("keys"),
        F.sum(F.when(F.col(key_col).isNull(),1).otherwise(0)).alias("null_keys"),
    ).first()

    ok = (
        int(p["rows"]) > 0
        and int(p["rows"]) == int(p["keys"])
        and int(p["null_keys"] or 0) == 0
    )
    if not ok:
        dimension_failures.append(object_name)
    dimension_rows.append(
        (object_name, int(p["rows"]), int(p["keys"]),
         int(p["null_keys"] or 0), "PASS" if ok else "FAIL")
    )

display(spark.createDataFrame(
    dimension_rows,
    ["dimension","rows","distinct_keys","null_keys","status"],
))

## 4. Fact grain and foreign-key validation

In [ ]:
fact_failures = []
fact_rows = []

for object_name, key_col in FACTS.items():
    table_name = f"{target_schema_fqn}.{object_name}"
    df = spark.table(table_name)
    p = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(key_col).alias("keys"),
        F.sum(F.when(F.col(key_col).isNull(),1).otherwise(0)).alias("null_keys"),
    ).first()

    ok = (
        int(p["rows"]) > 0
        and int(p["rows"]) == int(p["keys"])
        and int(p["null_keys"] or 0) == 0
    )
    if not ok:
        fact_failures.append(object_name)
    fact_rows.append(
        (object_name, int(p["rows"]), int(p["keys"]),
         int(p["null_keys"] or 0), "PASS" if ok else "FAIL")
    )

display(spark.createDataFrame(
    fact_rows,
    ["fact","rows","distinct_keys","null_keys","status"],
))

enc = spark.table(f"{target_schema_fqn}.fact_encounters")
cond = spark.table(f"{target_schema_fqn}.fact_conditions")

fk_checks = {
    "encounter_date": enc.filter(F.col("date_key").isNull()).count(),
    "encounter_patient": enc.filter(F.col("patient_key").isNull()).count(),
    "encounter_organization": enc.filter(F.col("organization_key").isNull()).count(),
    "encounter_payer": enc.filter(F.col("payer_key").isNull()).count(),
    "encounter_provider": enc.filter(
        F.col("provider_id").isNotNull() & F.col("provider_key").isNull()
    ).count(),
    "condition_patient": cond.filter(F.col("patient_key").isNull()).count(),
    "condition_dimension": cond.filter(F.col("condition_key").isNull()).count(),
    "condition_date": cond.filter(F.col("condition_start_date_key").isNull()).count(),
    "condition_encounter": cond.filter(
        F.col("encounter_id").isNotNull() & F.col("encounter_key").isNull()
    ).count(),
}

fk_failures = [n for n, count in fk_checks.items() if count > 0]

display(spark.createDataFrame(
    [(n, int(c), "PASS" if c == 0 else "FAIL") for n, c in fk_checks.items()],
    ["foreign_key","missing_rows","status"],
))

## 5. Aggregate grains and reconciliation

In [ ]:
aggregate_failures = []
aggregate_rows = []

for object_name, keys in AGGREGATES.items():
    df = spark.table(f"{target_schema_fqn}.{object_name}")
    rows = df.count()
    dupes = df.groupBy(*keys).count().filter(F.col("count") > 1).count()
    ok = rows > 0 and dupes == 0
    if not ok:
        aggregate_failures.append(object_name)
    aggregate_rows.append(
        (object_name, rows, dupes, "PASS" if ok else "FAIL")
    )

display(spark.createDataFrame(
    aggregate_rows,
    ["aggregate","rows","duplicate_grain_rows","status"],
))

encounter_count = enc.count()
condition_count = cond.count()

daily_count = int(
    spark.table(f"{target_schema_fqn}.agg_daily_encounters")
    .agg(F.sum("encounter_count").alias("n"))
    .first()["n"] or 0
)
org_count = int(
    spark.table(f"{target_schema_fqn}.agg_organization_performance")
    .agg(F.sum("encounter_count").alias("n"))
    .first()["n"] or 0
)
payer_count = int(
    spark.table(f"{target_schema_fqn}.agg_payer_performance")
    .agg(F.sum("encounter_count").alias("n"))
    .first()["n"] or 0
)
condition_agg_count = int(
    spark.table(f"{target_schema_fqn}.agg_condition_summary")
    .agg(F.sum("condition_events").alias("n"))
    .first()["n"] or 0
)

reconciliation = {
    "daily_to_fact_encounters": daily_count == encounter_count,
    "organization_to_fact_encounters": org_count == encounter_count,
    "payer_to_fact_encounters": payer_count == encounter_count,
    "condition_summary_to_fact_conditions": condition_agg_count == condition_count,
}

reconciliation_failures = [n for n, ok in reconciliation.items() if not ok]

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in reconciliation.items()],
    ["reconciliation","status"],
))

## 6. Business sanity and final summary

In [ ]:
business_checks = {
    "encounter_duration_non_negative":
        enc.filter(F.col("duration_minutes") < 0).count() == 0,
    "claim_cost_non_negative":
        enc.filter(F.col("total_claim_cost") < 0).count() == 0,
    "payer_coverage_non_negative":
        enc.filter(F.col("payer_coverage") < 0).count() == 0,
    "condition_duration_non_negative":
        cond.filter(F.col("condition_duration_days") < 0).count() == 0,
    "daily_encounter_count_positive":
        spark.table(f"{target_schema_fqn}.agg_daily_encounters")
        .filter(F.col("encounter_count") <= 0).count() == 0,
}

business_failures = [n for n, ok in business_checks.items() if not ok]

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in business_checks.items()],
    ["business_check","status"],
))

final_checks = [
    ("required_tables", len(missing_tables) == 0),
    ("external_locations", len(location_failures) == 0),
    ("dimension_keys", len(dimension_failures) == 0),
    ("fact_grains", len(fact_failures) == 0),
    ("foreign_keys", len(fk_failures) == 0),
    ("aggregate_grains", len(aggregate_failures) == 0),
    ("aggregate_reconciliation", len(reconciliation_failures) == 0),
    ("business_sanity", len(business_failures) == 0),
]

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in final_checks],
    ["validation_area","status"],
))

failed = [n for n, ok in final_checks if not ok]
if run_validation and failed:
    raise RuntimeError("Lab 06 final validation failed: " + ", ".join(failed))

print("LAB 06 EXTERNAL V2 — FINAL GOLD VALIDATION COMPLETE")
print("Recurring Gold pipeline status: PASS")
print("Validated: 6 dimensions, 2 facts, 4 aggregates")
print("Serverless compatibility: PASS")
print("REFRESH TABLE calls: 0")